# Notebook 11 — Training Persian GPT

This notebook trains the Persian GPT model end-to-end.

It includes:
- DataLoader / Dataset
- Model instantiation
- Optimizer + Scheduler
- AMP (mixed precision)
- TF32 + cuDNN benchmark
- Gradient clipping
- Parameter / model size reporting
- GPU / VRAM / power / temperature monitoring
- Tokens/sec + ETA
- CSV logging + loss history
- Checkpoint save/load (`latest.pt`, `best.pt`, `epoch_xxx.pt`)
- Resume training

> **Prerequisite:** move the `GPTModel` class built in Notebook 07 into `src/model.py` (a `GPTModel` class exported from that module) before running this notebook. This keeps the notebook copy-paste-ready for `train.py`.

## 1. Imports

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import os
import csv
import time
import math
import json
from pathlib import Path
from dataclasses import asdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import sentencepiece as spm

from configs.config import GPTConfig
from src.dataset import PersianDataset
from src.model import PersianGPT


## 2. Reproducibility, Device, TF32, cuDNN Benchmark

All values come from `GPTConfig`, no magic numbers.

In [3]:
cfg = GPTConfig()

def set_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")

# TF32 (safe speedup on Ampere+/Ada GPUs, RTX 4060 included)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# cuDNN autotuner — good for fixed input shapes (fixed seq_len here)
torch.backends.cudnn.benchmark = True

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


c:\Users\FH\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\backends\__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  self.setter(val)


## 3. Config Sanity Check

In [5]:
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False, default=str))

{
  "vocab_size": 32000,
  "max_seq_len": 256,
  "embed_dim": 256,
  "num_heads": 8,
  "num_layers": 6,
  "ffn_dim": 1024,
  "dropout": 0.1,
  "batch_size": 16,
  "learning_rate": 0.0003,
  "weight_decay": 0.01,
  "epochs": 10,
  "tokenizer_path": "C:\\Users\\FH\\Documents\\PersionLLM\\PersianGPT\\data\\tokenizer\\persian.model",
  "train_file": "C:\\Users\\FH\\Documents\\PersionLLM\\PersianGPT\\data\\processed\\clean_corpus.txt",
  "checkpoint_dir": "C:\\Users\\FH\\Documents\\PersionLLM\\PersianGPT\\checkpoints",
  "device": "cuda"
}


## 4. Tokenizer

Loaded only to confirm vocabulary size / special tokens match `cfg`.

In [7]:
sp = spm.SentencePieceProcessor()
sp.load(str(cfg.tokenizer_path))

assert sp.get_piece_size() == cfg.vocab_size, (
    f"Tokenizer vocab size ({sp.get_piece_size()}) does not match "
    f"cfg.vocab_size ({cfg.vocab_size})"
)

PAD_ID = sp.pad_id() if sp.pad_id() != -1 else 3
BOS_ID = sp.bos_id() if sp.bos_id() != -1 else 1
EOS_ID = sp.eos_id() if sp.eos_id() != -1 else 2
UNK_ID = sp.unk_id() if sp.unk_id() != -1 else 0

print(f"vocab_size={sp.get_piece_size()}  PAD={PAD_ID}  BOS={BOS_ID}  EOS={EOS_ID}  UNK={UNK_ID}")


vocab_size=32000  PAD=3  BOS=1  EOS=2  UNK=0


## 5. Load Token Stream

`cfg.train_file` can point either to:
- a pre-tokenized `.npy` file (`int32`/`int16` array of token ids), or
- the raw cleaned text corpus, which is tokenized once here and cached next to it as `<name>_tokens.npy` for fast reload on future runs.

In [8]:
def load_token_array(cfg: GPTConfig, sp: spm.SentencePieceProcessor) -> np.ndarray:
    train_path = Path(cfg.train_file)

    if train_path.suffix == ".npy":
        tokens = np.load(train_path, mmap_mode="r")
        print(f"Loaded pre-tokenized array: {train_path}  ({len(tokens):,} tokens)")
        return tokens

    cache_path = train_path.with_name(train_path.stem + "_tokens.npy")
    if cache_path.exists():
        tokens = np.load(cache_path, mmap_mode="r")
        print(f"Loaded cached token array: {cache_path}  ({len(tokens):,} tokens)")
        return tokens

    print(f"No cached tokens found. Tokenizing {train_path} ...")
    ids: list[int] = []
    with open(train_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ids.extend(sp.encode(line, out_type=int))
            ids.append(EOS_ID)

    tokens = np.array(ids, dtype=np.int32)
    np.save(cache_path, tokens)
    print(f"Tokenized and cached {len(tokens):,} tokens -> {cache_path}")
    return tokens


token_array = load_token_array(cfg, sp)


No cached tokens found. Tokenizing C:\Users\FH\Documents\PersionLLM\PersianGPT\data\processed\clean_corpus.txt ...


KeyboardInterrupt: 

## 6. Dataset & DataLoader

In [ ]:
train_dataset = PersianDataset(tokens=token_array, seq_len=cfg.max_seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)

# Rough steps-per-epoch estimate for an IterableDataset with random sampling
steps_per_epoch = len(token_array) // (cfg.batch_size * cfg.max_seq_len)
print(f"Approx steps per epoch: {steps_per_epoch:,}")


## 7. Model

In [ ]:
model = GPTModel(cfg).to(device)
print(model)


## 8. Parameter Counter & Model Size

In [ ]:
def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def model_size_mb(model: nn.Module) -> float:
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024 ** 2)


total_params, trainable_params = count_parameters(model)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size:            {model_size_mb(model):.2f} MB")


## 9. Optimizer, Scheduler, AMP

In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
)

total_steps = cfg.max_steps if cfg.max_steps else steps_per_epoch * cfg.epochs

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
)

scaler = GradScaler(device="cuda", enabled=(device.type == "cuda"))

print(f"Total training steps: {total_steps:,}")


## 10. GPU / VRAM / Power / Temperature Monitor

Uses `pynvml` when available, otherwise falls back to `torch.cuda` memory stats only.

In [ ]:
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    _NVML_AVAILABLE = True
except Exception:
    _NVML_AVAILABLE = False
    print("pynvml not available — GPU power/temperature/utilization will be reported as N/A.")


def get_gpu_stats(device: torch.device) -> dict:
    stats = {
        "vram_used_mb": 0.0,
        "vram_reserved_mb": 0.0,
        "gpu_util_pct": None,
        "gpu_power_w": None,
        "gpu_temp_c": None,
    }
    if device.type != "cuda":
        return stats

    stats["vram_used_mb"] = torch.cuda.memory_allocated(device) / (1024 ** 2)
    stats["vram_reserved_mb"] = torch.cuda.memory_reserved(device) / (1024 ** 2)

    if _NVML_AVAILABLE:
        util = pynvml.nvmlDeviceGetUtilizationRates(_nvml_handle)
        power_mw = pynvml.nvmlDeviceGetPowerUsage(_nvml_handle)
        temp_c = pynvml.nvmlDeviceGetTemperature(_nvml_handle, pynvml.NVML_TEMPERATURE_GPU)
        stats["gpu_util_pct"] = util.gpu
        stats["gpu_power_w"] = power_mw / 1000.0
        stats["gpu_temp_c"] = temp_c

    return stats


## 11. Checkpoint Utilities

Writes `latest.pt` every `cfg.save_every_epoch` epochs, `best.pt` whenever the epoch loss improves, and periodic `epoch_xxx.pt` snapshots.

In [ ]:
checkpoint_dir = Path(cfg.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)


def save_checkpoint(path: Path, model, optimizer, scheduler, scaler,
                     epoch: int, global_step: int, best_loss: float,
                     loss_history: list[float], cfg: GPTConfig) -> None:
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "best_loss": best_loss,
        "loss_history": loss_history,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "config": asdict(cfg),
    }, path)


def load_checkpoint(path: Path, model, optimizer, scheduler, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    return {
        "epoch": ckpt["epoch"],
        "global_step": ckpt["global_step"],
        "best_loss": ckpt["best_loss"],
        "loss_history": ckpt["loss_history"],
    }


## 12. CSV Logger

In [ ]:
class CSVLogger:
    def __init__(self, log_dir: str, filename: str = "train_log.csv"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.path = self.log_dir / filename
        self._fieldnames = [
            "epoch", "step", "global_step", "loss", "lr",
            "tokens_per_sec", "eta_min",
            "vram_used_mb", "vram_reserved_mb",
            "gpu_util_pct", "gpu_power_w", "gpu_temp_c",
        ]
        is_new = not self.path.exists()
        self._file = open(self.path, mode="a", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(self._file, fieldnames=self._fieldnames)
        if is_new:
            self._writer.writeheader()
            self._file.flush()

    def log(self, **kwargs) -> None:
        self._writer.writerow(kwargs)
        self._file.flush()

    def close(self) -> None:
        self._file.close()


csv_logger = CSVLogger(log_dir="logs")


## 13. Resume Training

In [ ]:
start_epoch = 0
global_step = 0
best_loss = float("inf")
loss_history: list[float] = []

latest_ckpt_path = checkpoint_dir / "latest.pt"

if cfg.resume and latest_ckpt_path.exists():
    state = load_checkpoint(latest_ckpt_path, model, optimizer, scheduler, scaler, device)
    start_epoch = state["epoch"] + 1
    global_step = state["global_step"]
    best_loss = state["best_loss"]
    loss_history = state["loss_history"]
    print(f"Resumed from {latest_ckpt_path} — starting at epoch {start_epoch}, "
          f"global_step {global_step}, best_loss {best_loss:.4f}")
else:
    print("Starting training from scratch.")


## 14. Training Loop

In [ ]:
def format_eta(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def train(cfg: GPTConfig, model, train_loader, optimizer, scheduler, scaler,
          device, start_epoch: int, global_step: int, best_loss: float,
          loss_history: list[float]):

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    model.train()

    training_start = time.time()
    stop_training = False

    for epoch in range(start_epoch, cfg.epochs):
        epoch_start = time.time()
        running_loss = 0.0
        running_tokens = 0
        step_in_epoch = 0

        for input_ids, target_ids in train_loader:
            step_start = time.time()

            input_ids = input_ids.to(device, non_blocking=True)
            target_ids = target_ids.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(input_ids)
                loss = criterion(
                    logits.view(-1, cfg.vocab_size),
                    target_ids.view(-1),
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            step_time = time.time() - step_start
            batch_tokens = input_ids.numel()
            tokens_per_sec = batch_tokens / max(step_time, 1e-8)

            running_loss += loss.item()
            running_tokens += batch_tokens
            step_in_epoch += 1
            global_step += 1

            if global_step % cfg.log_interval == 0:
                avg_loss = running_loss / step_in_epoch
                elapsed = time.time() - training_start
                steps_done = (epoch - start_epoch) * steps_per_epoch + step_in_epoch
                steps_total = (cfg.epochs - start_epoch) * steps_per_epoch
                steps_remaining = max(steps_total - steps_done, 0)
                eta_seconds = steps_remaining * (elapsed / max(steps_done, 1))

                gpu_stats = get_gpu_stats(device)
                current_lr = scheduler.get_last_lr()[0]

                print(
                    f"Epoch {epoch+1}/{cfg.epochs} | Step {step_in_epoch}/{steps_per_epoch} "
                    f"| GlobalStep {global_step} | Loss {loss.item():.4f} "
                    f"| AvgLoss {avg_loss:.4f} | LR {current_lr:.2e} "
                    f"| Tok/s {tokens_per_sec:,.0f} | ETA {format_eta(eta_seconds)} "
                    f"| VRAM {gpu_stats['vram_used_mb']:.0f}MB "
                    f"| Util {gpu_stats['gpu_util_pct']}% "
                    f"| Power {gpu_stats['gpu_power_w']}W "
                    f"| Temp {gpu_stats['gpu_temp_c']}C"
                )

                csv_logger.log(
                    epoch=epoch + 1,
                    step=step_in_epoch,
                    global_step=global_step,
                    loss=loss.item(),
                    lr=current_lr,
                    tokens_per_sec=round(tokens_per_sec, 2),
                    eta_min=round(eta_seconds / 60, 2),
                    vram_used_mb=round(gpu_stats["vram_used_mb"], 2),
                    vram_reserved_mb=round(gpu_stats["vram_reserved_mb"], 2),
                    gpu_util_pct=gpu_stats["gpu_util_pct"],
                    gpu_power_w=gpu_stats["gpu_power_w"],
                    gpu_temp_c=gpu_stats["gpu_temp_c"],
                )

            if cfg.max_steps and global_step >= cfg.max_steps:
                stop_training = True
                break

        epoch_loss = running_loss / max(step_in_epoch, 1)
        loss_history.append(epoch_loss)
        epoch_time = time.time() - epoch_start
        print(f"\n=== Epoch {epoch+1} finished | AvgLoss {epoch_loss:.4f} "
              f"| Time {format_eta(epoch_time)} ===\n")

        if cfg.save_every_epoch and (epoch + 1) % cfg.save_every_epoch == 0:
            save_checkpoint(
                checkpoint_dir / "latest.pt", model, optimizer, scheduler, scaler,
                epoch, global_step, best_loss, loss_history, cfg,
            )
            save_checkpoint(
                checkpoint_dir / f"epoch_{epoch+1:03d}.pt", model, optimizer,
                scheduler, scaler, epoch, global_step, best_loss, loss_history, cfg,
            )
            print(f"Saved checkpoint: latest.pt, epoch_{epoch+1:03d}.pt")

        if cfg.save_best_model and epoch_loss < best_loss:
            best_loss = epoch_loss
            save_checkpoint(
                checkpoint_dir / "best.pt", model, optimizer, scheduler, scaler,
                epoch, global_step, best_loss, loss_history, cfg,
            )
            print(f"New best model saved (loss={best_loss:.4f})")

        if stop_training:
            print("Reached cfg.max_steps — stopping training.")
            break

    # Always persist final state as latest.pt
    save_checkpoint(
        checkpoint_dir / "latest.pt", model, optimizer, scheduler, scaler,
        epoch, global_step, best_loss, loss_history, cfg,
    )

    return loss_history, best_loss, global_step


## 15. Run Training

In [ ]:
loss_history, best_loss, global_step = train(
    cfg, model, train_loader, optimizer, scheduler, scaler,
    device, start_epoch, global_step, best_loss, loss_history,
)

csv_logger.close()
print(f"Training complete. Best loss: {best_loss:.4f}")
